In [ ]:
from dataclasses import dataclass
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import rasterio as rio
from scipy.stats import mannwhitneyu
import seaborn as sns
import torch
from torch import Tensor
from tqdm import tqdm
import xarray as xr

from project.nn.streamflow_net import StreamflowNet
from project.utils.data import enrich, load_hillshade, sel_subset
from project.utils.metrics import compute_metrics
from project.utils.plotting import ProjectCustomExtent, despine
from project.utils.unit_conversion import convert_xr_m3s_to_mmd

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr('/net/argon/landclim/kraftb/machflow_routing/data/results/predictions.zarr').sel(
    time=slice('1993', None))

# Load HRU polygons.
hrus = gpd.read_file(
    '/net/argon/landclim/kraftb/machflow_routing/data/processed/hrus/hrus.gpkg', use_arrow=True).set_index('EZGNR')

metrics_max = ['kge', 'nse', 'nnse', 'r']
metrics_min = ['rmse', 'bias2', 'varerr', 'phaseerr', 'relbias', 'flv', 'fhv']

metric_nice_names = {
    'bias2': 'Bias²',
    'varerr': 'Variance error',
    'phaseerr': 'Phase error',
    'nse': 'Nash-Sutcliffe Efficiency',
    'nnse': 'Normalized Nash-Sutcliffe Efficiency',
    'kge': 'Kling-Gupta Efficiency',
    'r': 'Pearson\'s r',
    'relbias': '%Bias',
    'flv': '%Low Flow Bias (<Q5)',
    'fhv': '%High Flow Bias (>Q98)',
}

# Compute metrics and add them to dataset.
met = compute_metrics(ds.Q_obs_m3s, ds.Q_pred_m3s, metrics=metrics_min + metrics_max, dim='time')
ds = xr.merge([ds, met])

for m in metrics_min + metrics_max:
    name = f'{m}_impr'

    # Create the target variable once per metric.
    # Use the metric itself as a template (and make sure it's float so NaN works).
    if name not in ds:
        ds[name] = xr.full_like(ds[m], np.nan, dtype=float).load()

    for model in ds.model.values:
        # Pair model with the corresponding other model
        model_type, split_type = model.split('_')
        other_model_type = 'routed' if model_type == 'lumped' else 'lumped'
        other_model = f'{other_model_type}_{split_type}'
        if other_model not in ds.model.values:
            continue

        value_model = ds.sel(model=model)[m]
        value_other = ds.sel(model=other_model)[m]

        if m in metrics_min:
            value_model = np.abs(value_model)
            value_other = np.abs(value_other)

        if m in metrics_min:
            # Lower is better
            pec_impr = (value_other - value_model) / (value_other + 1e-6) * 100.0
        else:
            # Higher is better
            pec_impr = (value_model - value_other) / (1 - value_other) * 100.0

        # Write only the slice for this model
        ds[name].loc[dict(model=model)] = pec_impr


ds['R_pred_mean'] = ds['R_pred'].mean('time').compute()

In [ ]:
rivers = gpd.read_file('/net/argon/landclim/kraftb/machflow_routing/data/raw/rivernet', use_arrow=True)

In [ ]:
bg = load_hillshade(
    hillshade_dir='/net/argon/landclim/kraftb/machflow_routing/data/processed/',
    hrus=hrus,
    ds=ds,
    use_terrain=False,
    target_crs='EPSG:2056',
    force_recompute=False)

# Retention sensitivities

In [ ]:
snet = StreamflowNet.load_from_checkpoint(
    '/net/argon/landclim/kraftb/machflow_routing/logs/routed/random/tune/trial_018/checkpoints/best.ckpt', runoff_net_exp_dir='/net/argon/landclim/kraftb/machflow_routing/logs/pretraining/', map_location='cpu'
)

In [ ]:
snet.eval()
tnet = snet.routing_net.tnet

@dataclass
class Batch:
    """Dummy batch structure.
    
    Note that only the x_routing field is used as we won't subset the graph, i.e., all other fields are dummies
    required for (yet ignored by) spatial encoding.
    """
    x_routing: Tensor
    k_hops: Tensor = torch.ones(1, dtype=torch.int)
    num_nodes: int = 1
    k_edge_index: tuple[Tensor, Tensor] = (torch.zeros(2, 0, dtype=torch.long), torch.zeros(0, dtype=torch.float))
    k: Tensor = torch.zeros(1, 1, dtype=torch.float)

routing_fields =[
    'area',
    'dam_height',
    'impoundment_volume',
    'is_barrage',
    'is_floodcontrol',
    'is_hydroelectricity',
    'is_lake',
    'is_other_aim',
    'is_regulated_lake',
    'is_storage_dam',
    'slope_mean',
    'upstream_area',
    'water_freq'
]

routing_avg = [
    0 if np.issubdtype(ds[field].dtype, bool) else ds[field].mean().compute().item() for field in routing_fields
]
routing_rng = [
    (0, 1) if np.issubdtype(ds[field].dtype, bool) else (
        ds[field].min().compute().item(), ds[field].load().max().compute().item() if field in ('dam_height', 'impoundment_volume') else ds[field].load().quantile(0.95).compute().item()
    ) for field in routing_fields
]

default_batch = Batch(x_routing=torch.tensor([routing_avg], dtype=torch.float))

In [ ]:
def plot_routing_sensitivity(**change_default) -> None:

    if len(change_default) > 0:
        routing_avg_new = routing_avg.copy()
        for k, v in change_default.items():
            field_id = routing_fields.index(k)
            routing_avg_new[field_id] = v
    else:
        routing_avg_new = routing_avg.copy()

    fig, axes = plt.subplots(1, len(routing_fields), figsize=(len(routing_fields) * 1.5, 1), sharey=True)

    for i, field in enumerate(routing_fields):

        ax = axes[i]

        field_id = routing_fields.index(field)
        field_iter_values = torch.linspace(routing_rng[field_id][0], routing_rng[field_id][1], steps=100)

        field_x_routing = []

        for v in field_iter_values:
            x_routing = torch.tensor([routing_avg_new], dtype=torch.float)
            x_routing[0, field_id] = v
            field_x_routing.append(x_routing)

        field_x_routing = torch.cat(field_x_routing, dim=0)
        field_batch = Batch(x_routing=field_x_routing.float())
        default_d = torch.zeros(field_batch.x_routing.shape[0], 1, dtype=torch.float)

        r = []
        d_range = np.logspace(0, 3, num=1000, base=10.0)

        with torch.no_grad():
            x_enc, _ = tnet.spatial_encode(
                batch=field_batch  # type: ignore
            )

            for d in d_range:
                s = default_d.clone() + d
                d_enc = tnet.dynamic_encode(x_enc=x_enc, d=s)
                r.append(tnet.final_transform(d_enc).squeeze())

        r = torch.stack(r, dim=0)

        r_vmax = r.max().item()

        cmap = plt.get_cmap('RdPu')

        if r_vmax == 1:
            r_max = 0.99
            cmap.set_over("red")

        mesh = ax.pcolormesh(
            field_iter_values,               # x‑edges
            d_range,               # y‑edges (log‑spaced!)
            r,                   # note the transpose: pcolormesh expects (Ny, Nx)
            shading='auto',        # fills each quadrilateral
            cmap=cmap,
            vmin=0.0,
            vmax=r_vmax,
        )

        axhist = ax.inset_axes([0.0, 1.0, 1.0, 0.3])
        # ds[field].plot.hist(bins=field_iter_values, density=True, ax=axhist, ec='none', fc='k')
        sns.kdeplot(ds[field].values,
            bw_adjust=0.5, fill=True, alpha=0.5, ec='k', fc='0.3',
            ax=axhist, clip=(routing_rng[field_id][0], routing_rng[field_id][1]))
        axhist.sharex(ax)
        despine(axhist, which='all')
        # Remove all ticks
        axhist.set_yticks([])
        axhist.set_ylabel('')
        axhist.set_xlabel('')
        axhist.tick_params(axis='x', which='both',          # axis='x' → affect only x‑ticks
                            bottom=False, top=False,      # no tick marks
                            labelbottom=False)

        axins = ax.inset_axes([0.1, -0.75, 0.8, 0.1])
        cbar = fig.colorbar(mesh, ax=ax, orientation='horizontal', cax=axins, extend='max', label='Retention')
        # axins.xaxis.set_ticks_position('top')
        # axins.set_title('Retention', fontsize=8)
        ax.set_yscale('log')
        ax.set_xlabel(field.replace('_', ' ').capitalize(), fontsize=8)
        ax.set_title('')
        if i == 0:
            ax.set_ylabel('Storage (m³/s)', fontsize=8)
        else:
            ax.set_ylabel('')

        cbar.ax.xaxis.set_minor_locator(mpl.ticker.LogLocator(subs=np.arange(2, 9) * 0.1))
        cbar.ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
        cbar.ax.tick_params(which='major', labelsize=5)

    title = 'Routing sensitivity analysis'

    if len(change_default) > 0:
        title += f' with {", ".join([f"{k}={v}" for k, v in change_default.items()])}'

    fig.suptitle(title, y=1.35)


In [ ]:
plot_routing_sensitivity()

In [ ]:
plot_routing_sensitivity(water_freq=1)

In [ ]:
plot_routing_sensitivity(is_lake=1)

In [ ]:
plot_routing_sensitivity(water_freq=1, is_lake=1)

In [ ]:
plot_routing_sensitivity(is_barrage=1, is_hydroelectricity=1)

In [ ]:
plot_routing_sensitivity(is_barrage=1, is_regulated_lake=1)

## Case study

In [ ]:
def plpt_ts(ds, ax, time_start, time_end, pred_labels):
    ds_s = ds.sel(time=slice(time_start, time_end))
    ds_s.Q_obs_m3s.sel(
        time=slice(time_start, time_end)).plot(
            ax=ax,
            color='0.0',
            lw=1.0,
            label='Observed'
        )

    for i, model in enumerate(['routed_random', 'lumped_random']):
        color = ["#DAA507", "#1E88E5"][i]

        ds_s.Q_pred_m3s.sel(model=model).plot(
            ax=ax,
            label=pred_labels[i],
            alpha=1.0,
            lw=1.2,
            ls='--',
            dashes=[3, 1],
            color=color,
            zorder=5-i
        )


fig = plt.figure(figsize=(10, 5))

gs = mpl.gridspec.GridSpec(2, 2, figure=fig, wspace=0.1, hspace=0.25)

ds_aare = ds.sel(hru=145830)

ax_tl = fig.add_subplot(gs[0, 0])
ax_tr = fig.add_subplot(gs[0, 1], sharey=ax_tl)
ax_b = fig.add_subplot(gs[1, :])

# z0 = ('2020-01-01', '2022-12-31')
# z1 = ('2020-05-01', '2020-12-31')
# z2 = ('2021-05-01', '2021-12-31')

z0 = ('2006-01-01', '2008-12-31')
z1 = ('2006-05-01', '2006-12-31')
z2 = ('2007-05-01', '2007-12-31')

routed_random_kge = ds_aare.kge.sel(model='routed_random').item()
lumped_random_kge = ds_aare.kge.sel(model='lumped_random').item()

labels = [
    f'DRAIN (distributed; KGE={routed_random_kge:.2f})',
    f'LSTM (lumped; KGE={lumped_random_kge:.2f})'
]

plpt_ts(ds_aare, ax_b, *z0, labels)
plpt_ts(ds_aare, ax_tl, *z1, labels)
plpt_ts(ds_aare, ax_tr, *z2, labels)

# for i, model in enumerate(['routed_random', 'lumped_random']):
#     color = ['tab:red', 'tab:blue'][i]
#     Q_pred_m3s.sel(model=model).plot(ax=ax_b, label=model.replace('_', ' ').title(), lw=0.7, ls='--', alpha=0.8, color=color)

# Add the connection patches

for s, (zoom_start, zoom_end) in enumerate([z1, z2]):
    top_axis = [ax_tl, ax_tr][s]

    zoom_start = pd.Timestamp(zoom_start)
    zoom_end = pd.Timestamp(zoom_end)

    ax_b.axvspan(zoom_start, zoom_end, ec='k', fc='0.92', lw=0.4, ls='--')

    for z, zoom in enumerate([zoom_start, zoom_end]):

        fig.add_artist(mpl.patches.ConnectionPatch(
            xyA=(z, 0), coordsA=top_axis.transAxes,
            xyB=(mpl.dates.date2num(zoom), ax_b.get_ylim()[1]), coordsB=ax_b.transData,
            color='k',
            lw=0.4,
            ls='--',
            zorder=0,
        ))

for i, ax in enumerate([ax_tl, ax_tr, ax_b]):
    ax.set_xlabel('')
    despine(ax)
    ax.set_title('')

    if i != 1:
        ax.set_ylabel('Discharge (m$^3$ s$^{-1}$)')
    else:
        ax.set_ylabel('')

ax_b.legend(loc='upper right', bbox_to_anchor=(1, 1), frameon=False)

for i, ax in enumerate([ax_tr, ax_b]):
    event = pd.Timestamp('2007-08-09')
    if i == 0:
        post_event = pd.Timestamp('2007-08-14')
    else:
        post_event = pd.Timestamp('2007-08-19')

    ax.axvline(mpl.dates.date2num(event), color='k', lw=0.6, ls='--')
    ax.text(
        mpl.dates.date2num(post_event),
        ax.get_ylim()[1] * 0.9,
        'Peak flood event',
        color='k',
        fontsize=6,
        rotation=90,
        va='top',
        ha='left',
        path_effects=[mpe.withStroke(linewidth=1, foreground="w")]
    )

fig.align_ylabels([ax_tl, ax_b])

fig.savefig('./figures/time_series.png', dpi=300, bbox_inches='tight')

# Contribution analysis

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
hru = 145830

In [ ]:
def select_dayofyear_window(ds: xr.Dataset | xr.DataArray, date: str, window: int) -> xr.Dataset | xr.DataArray:
    center_doy = ds.sel(time=date).time.dt.dayofyear.item()
    query_days = np.arange(center_doy - window, center_doy + window + 1)
    return ds.sel(time=ds.time.dt.dayofyear.isin(query_days))

dss = xr.open_zarr('/net/argon/landclim/kraftb/machflow_routing/logs/routed/random/source_contrib_145830.zarr')

In [ ]:
ds_aare = ds.sel(hru=hru, model='routed_random')

In [ ]:
source_ref = select_dayofyear_window(dss.q_source_m3s, date='2007-08-09', window=10).mean('time').compute()
source_day = dss.q_source_m3s.sel(time='2007-08-09').compute()
source_diff = source_day - source_ref

In [ ]:
source = xr.Dataset({
    'ref_m3s': source_ref,
    'day_m3s': source_day,
    'diff_m3s': source_diff,
    'area': dss.area,
})
convert_xr_m3s_to_mmd(source, variables=['ref_m3s', 'day_m3s', 'diff_m3s']).compute()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 5), sharey=True)

cbar_kwargs = {
    'label': 'Contribution on log scale (m³/s)',
    'orientation': 'horizontal',
}

source.ref_m3s.plot.imshow(
    interpolation='none',
    norm=mpl.colors.LogNorm(vmin=1e-5),
    cmap='magma_r',
    ax=axes[0],
    cbar_kwargs=cbar_kwargs)
axes[0].set_title('Historic average')
source.day_m3s.plot.imshow(
    interpolation='none',
    norm=mpl.colors.LogNorm(vmin=1e-5),
    cmap='magma_r',
    ax=axes[1],
    cbar_kwargs=cbar_kwargs)
axes[1].set_title('Contribution on 2021-07-15')

source.diff_m3s.plot.imshow(
    interpolation='none',
    norm=mpl.colors.SymLogNorm(vmin=-0.1, vmax=0.1, linthresh=0.001),
    cmap='RdBu_r',
    ax=axes[2],
    cbar_kwargs=cbar_kwargs
)
axes[2].set_title('Contribution anomaly')

fig.suptitle('Contribution of HRUs to outlet on 2021-07-15', fontsize=16)

In [ ]:
def enrich_with_grads(time_idx: int, var: str, agg_fn: str, consider_n_past: int = 7) -> tuple[gpd.GeoDataFrame, str]:
    time_start = ds_s.grad_time.isel(grad_time=max(time_idx - consider_n_past, 0)).dt.strftime('%Y-%m-%d').item()
    time_end = ds_s.grad_time.isel(grad_time=time_idx).dt.strftime('%Y-%m-%d').item()
    if 'grad_time' not in ds_s[var].dims:
        ds_subset = ds_s[var].sel(time=slice(time_start, time_end))
    else:
        ds_subset = ds_s[var].isel(grad_time=time_idx)

    if agg_fn == 'mean':
        ds_subset = ds_subset.mean('time')
    elif agg_fn == 'sum':
        ds_subset = ds_subset.sum('time')
    hrus_attr = enrich(
        ds=ds_subset,
        hrus=hrus.loc[ds_s.hru],
        variables=var
    )

    return hrus_attr, time_end

In [ ]:
hrus_source = enrich(
    source.day_mmd.sel(lag=0), hrus.loc[source.hru], variables='lag_instant')
hrus_source = enrich(
    source.day_mmd.sel(lag=slice(3, 1)).sum('lag'), hrus_source, variables='lag_1w')
hrus_source = enrich(
    source.day_mmd.sel(lag=slice(30, 4)).sum('lag'), hrus_source, variables='lag_remaining')

hrus_source = enrich(
    ds.prec.sel(time='2007-08-09', hru=hrus_source.index), hrus_source, variables='prec_instant')
hrus_source = enrich(
    ds.prec.sel(time=slice('2007-08-06', '2007-08-08'), hru=hrus_source.index).mean('time'), hrus_source, variables='prec_1w')
hrus_source = enrich(
    ds.prec.sel(time=slice('2007-07-10', '2007-08-05'), hru=hrus_source.index).mean('time'), hrus_source, variables='prec_remaining')

hrus_source = enrich(
    ds.q_pred_mmd.sel(time='2007-08-09', model='routed_random', hru=hrus_source.index), hrus_source, variables='q_instant')
hrus_source = enrich(
    ds.q_pred_mmd.sel(time=slice('2007-08-06', '2007-08-08'), model='routed_random', hru=hrus_source.index).mean('time'), hrus_source, variables='q_1w')
hrus_source = enrich(
    ds.q_pred_mmd.sel(time=slice('2007-07-10', '2007-08-05'), model='routed_random', hru=hrus_source.index).mean('time'), hrus_source, variables='q_remaining')

hrus_source = enrich(source.area, hrus_source, variables='area')

In [ ]:
convert_mmd_to_m3s(source.day_mmd.sum('lag').values, hrus_source['area'].values).sum()

In [ ]:
convert_mmd_to_m3s(source.day_mmd.isel(lag=slice(-30, None)).sum('lag').values, hrus_source['area'].values).sum()

In [ ]:
variables = [
    ['prec_instant', 'prec_1w', 'prec_remaining'],
    ['q_instant', 'q_1w', 'q_remaining'],
    ['lag_instant', 'lag_1w', 'lag_remaining'],
]
titles = [
    'August 9 (instant)',
    'August 6-8 (lag 1 to 3 days)',
    'July 10 to August 5 (lag 4 to 31 days)',
]
ylabels = [
    'Observed\nprecipitation (mean)',
    'Simulated\nrunoff (mean)',
    'Contribution to\nAugust 9 2007',
]

In [ ]:
from project.utils.unit_conversion import convert_mmd_to_m3s
from matplotlib.gridspec import GridSpec

In [ ]:
xmin, ymin, xmax, ymax = hrus_source.total_bounds
xrng = xmax - xmin
xmin -= xrng * 0.05
xmax += xrng * 0.05
yrng = ymax - ymin
ymin -= yrng * 0.05
ymax += yrng * 0.05
bounds = [xmin, xmax, ymin, ymax]

bg_bounds = bg.sel(lon=slice(*bounds[:2]), lat=slice(bounds[-1], bounds[-2])).load()

In [ ]:
fig = plt.figure(layout='constrained', figsize=(8, 4))

gs = GridSpec(3, 4, figure=fig, wspace=0.02, hspace=0.0, width_ratios=[1, 1, 1, 0.02])




for r in range(3):
    vmax = np.quantile(hrus_source[variables[r]], 0.98)
    norm = mpl.colors.Normalize(vmin=0, vmax=vmax)

    kwargs = {
        'cmap': ['GnBu', 'PuRd', 'BuPu'][r],
        'ec': 'k',
        'lw': 0.1,
        'alpha': 0.7,
        'legend': False,
    }

    for c in range(3):

        ax = fig.add_subplot(gs[r, c], projection=lv95)

        x = hrus_source.loc[hru].geometry.centroid.x
        y = hrus_source.loc[hru].geometry.centroid.y

        ax.set_extent(bounds, crs=lv95)
        # bg_bounds.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.8, zorder=0)

        rivers.loc[rivers.CUM_LEN > 20000].plot(ax=ax, color='k', lw=0.5, zorder=20, alpha=0.7)

        var = variables[r][c]

        hrus_source.plot(
            column=var,
            ax=ax,
            norm=norm,
            **kwargs
        )

        if c == 0:
            ax.text(-0.1, 0.5, ylabels[r], rotation=90, va='center', ha='center', transform=ax.transAxes, fontsize=8)
        
        if r == 0:
            ax.set_title(titles[c], fontsize=8)
        else:
            ax.set_title('')

        if r == 2:
            contr = convert_mmd_to_m3s(hrus_source[var].values, hrus_source['area'].values).sum()
            ax.text(
                0.015, 0.98, f'Total contribution:\n{contr:.0f} m$^3$ s$^{{-1}}$', transform=ax.transAxes,
                ha='left', va='top', fontsize=7, bbox=dict(facecolor='0.93', alpha=1, edgecolor='none', pad=1), zorder=100)

        ax.scatter(x, y, ec='k', fc='tab:red', s=12, lw=0.6, zorder=9999)

    cbar_ax = fig.add_subplot(gs[r, -1])
    cax = cbar_ax.inset_axes([0.0, 0.1, 1.0, 0.8])
    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=kwargs['cmap'])
    cbar = plt.colorbar(
        mappable,
        cax=cax,
        orientation='vertical',
        extend='max',
        label=['Precipitation (mm d$^{-1}$)', 'Runoff (mm d$^{-1}$)', 'Contribution (mm d$^{-1}$)'][r],
    )
    cbar.ax.yaxis.label.set_size(8)
    cbar.ax.tick_params(labelsize=8)
    cbar_ax.axis('off')

fig.savefig('./figures/case_study.png', dpi=300, bbox_inches='tight')